# Day 12 | HOL 1: Unity Catalog Permissions — UI and Code

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Calendar slot** | Day 12 · HOL 1 |
| **Builds on** | Day 11 ILT 4 (`Day11_4_ILT4_Data_Governance_Unity_Catalog.ipynb`) — that session taught real `GRANT`/`REVOKE` syntax but marked it explicitly **illustrative only, never executed**. Today it gets executed, for real, twice. |
| **Companion page** | `Day12_2_HOL1_Unity_Catalog_Permissions_UI_and_Code.html` — the UI steps (Phase 1) live there, not here, since clicking through Catalog Explorer can't be scripted |
| **Writes (practice only)** | `main.YOUR_SCHEMA.practice_customers` — the exact same practice table ILT 4 built. `GRANT`/`REVOKE` here run for real, but only ever against this table, never `gbmart.*` |
| **Reads (real, read-only)** | `SHOW GRANTS` against the real `gbmart` catalog/schema/table — Phase 3 |

### Learning Objectives
- Actually run a `GRANT` and a `REVOKE` — not just read the syntax
- Grant the exact same privilege two different ways (Catalog Explorer's Permissions tab, then SQL) and prove, via `SHOW GRANTS`, that both levers write to the *same* underlying state
- Read real grants on the real `gbmart` Gold layer, safely and read-only, and compare them to your own tiny practice grant

---
**Safety, upfront:** every `GRANT`/`REVOKE` that actually *runs* in this notebook targets only `main.YOUR_SCHEMA.practice_customers` — a table you own, that nobody else in the room is touching. The only thing this notebook does against the real `gbmart` catalog is `SHOW GRANTS` — non-destructive, metadata-only, read-only. Nothing here starts a job, workflow, pipeline, SQL warehouse, or cluster.

**Instructions:** Run Setup first. Then switch to the companion HTML page for Phase 1 (UI). Come back here for Phase 2 (Code) and Phase 3 (Compare to Real `gbmart`).

## Why This, Now

ILT 4 gave you the shape of `GRANT`/`REVOKE` and told you, explicitly, that nothing in that notebook actually executed one — the practice table was reserved for row filters and masks instead. That was a deliberate gap, not an oversight: permissions deserved their own hands-on, because "I can read this syntax" and "I just watched a grant appear and disappear because I ran a statement" are genuinely different levels of understanding. Today closes that gap, and adds the piece ILT 4 never touched at all — the Catalog Explorer **Permissions tab**, the UI path to the exact same outcome.

## Setup — Run This First

Builds the same `practice_customers` table ILT 4 used — if you still have that one around, this just confirms it's there; if this is a fresh session, it creates it from scratch. Either way, idempotent: safe to re-run.

In [ ]:
# --- Practice schema + table setup -- isolates this HOL from the real gbmart catalog --
# Replace YOUR_SCHEMA with something unique to you if you want a clean, personal copy
YOUR_SCHEMA = "main.YOUR_SCHEMA"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {YOUR_SCHEMA}")

PRACTICE_TABLE = f"{YOUR_SCHEMA}.practice_customers"
PRINCIPAL = "`account users`"   # the built-in Unity Catalog group meaning "everyone in this account" -- always exists, no admin setup needed

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {PRACTICE_TABLE} (
        customer_id   INT,
        customer_name STRING,
        region        STRING,
        email         STRING
    ) USING DELTA
""")

if spark.table(PRACTICE_TABLE).count() == 0:
    spark.sql(f"""
        INSERT INTO {PRACTICE_TABLE} VALUES
            (1, 'Asha Rao',   'US', 'asha.rao@example.com'),
            (2, 'Liam Chen',  'US', 'liam.chen@example.com'),
            (3, 'Priya Nair', 'IN', 'priya.nair@example.com'),
            (4, 'Tom Becker', 'UK', 'tom.becker@example.com'),
            (5, 'Wei Zhang',  'IN', 'wei.zhang@example.com'),
            (6, 'Sara Ahmed', 'US', 'sara.ahmed@example.com')
    """)

print(f"Practice table ready : {PRACTICE_TABLE}")
print(f"Rows                 : {spark.table(PRACTICE_TABLE).count()}")
print(f"Principal we'll use  : {PRINCIPAL}")

## → Now Go Do Phase 1 (UI)

Switch to `Day12_2_HOL1_Unity_Catalog_Permissions_UI_and_Code.html`, Phase 1. You'll click through Catalog Explorer to grant `SELECT` on `main.YOUR_SCHEMA.practice_customers` to `` `account users` `` — entirely with the mouse, no SQL.

**Come back here once that's done.**

## Phase 2 (Code) — Read What the UI Just Wrote

`SHOW GRANTS` is a read — it changes nothing. If Phase 1 worked, the grant you just clicked into existence via the UI should show up here, in code, immediately. This is the whole point: **the UI and SQL are two doors into the exact same room.**

In [ ]:
print(f"SHOW GRANTS ON TABLE {PRACTICE_TABLE} -- after your Phase 1 UI grant:")
spark.sql(f"SHOW GRANTS ON TABLE {PRACTICE_TABLE}").display()

### Now Revoke It — By Code This Time

Same privilege, same principal, same table — taken away with a `REVOKE` statement instead of a UI click.

In [ ]:
spark.sql(f"REVOKE SELECT ON TABLE {PRACTICE_TABLE} FROM {PRINCIPAL}")
print(f"Revoked SELECT on {PRACTICE_TABLE} from {PRINCIPAL}")
print()
print("SHOW GRANTS again -- the row from Phase 1 should be gone:")
spark.sql(f"SHOW GRANTS ON TABLE {PRACTICE_TABLE}").display()

### Now Grant It Back — By Code, Not the UI

The other direction, purely in SQL this time — exactly the syntax ILT 4 showed you as illustrative-only. Not illustrative anymore.

In [ ]:
spark.sql(f"GRANT SELECT ON TABLE {PRACTICE_TABLE} TO {PRINCIPAL}")
print(f"Granted SELECT on {PRACTICE_TABLE} to {PRINCIPAL} -- via code")
print()
print("SHOW GRANTS one more time -- should look identical to the state Phase 1's UI click produced:")
spark.sql(f"SHOW GRANTS ON TABLE {PRACTICE_TABLE}").display()

**What you just proved:** the same privilege, on the same object, for the same principal, appeared, disappeared, and reappeared — once by mouse click, twice by SQL statement — and `SHOW GRANTS` couldn't tell you which lever was used. Unity Catalog doesn't have a "UI version" of a grant and a "code version" of a grant. There's one privilege model; the Permissions tab and `GRANT`/`REVOKE` are just two different ways to reach into it.

## Phase 3 — Compare Against the Real `gbmart` Gold Layer (Read-Only)

Your practice grant above involves one table, one principal, one privilege. The real `gbmart` catalog has been accumulating real grants all course. Same `SHOW GRANTS` mechanism, now pointed at the real thing — wrapped in `try/except`, since this demo account may not be the owner/admin of `gbmart` (a normal Unity Catalog boundary, not a bug).

In [ ]:
CATALOG = "gbmart"

grant_checks = [
    ("CATALOG gbmart",               f"SHOW GRANTS ON CATALOG {CATALOG}"),
    ("SCHEMA gbmart.gold",           f"SHOW GRANTS ON SCHEMA {CATALOG}.gold"),
    ("TABLE gbmart.gold.fact_sales", f"SHOW GRANTS ON TABLE {CATALOG}.gold.fact_sales"),
]

for label, stmt in grant_checks:
    print(f"--- SHOW GRANTS ON {label} ---")
    try:
        spark.sql(stmt).display()
    except Exception as e:
        print(f"Could not read grants on {label} -- this account is probably not the "
              f"owner/admin of that object, which is a normal Unity Catalog boundary, "
              f"not a bug. ({str(e)[:120]})")
    print()

## Key Takeaways
- `GRANT`/`REVOKE` and the Catalog Explorer Permissions tab both write to the same one Unity Catalog privilege model — neither is more "real" than the other.
- `SHOW GRANTS` is the read-only way to check either one, on any object, at any time.
- Every write you actually ran targeted your own practice table — never `gbmart.*` — the same safety boundary ILT 4 established for row filters and masks.
- Further reading (not required to finish this HOL): Databricks' own ["Manage privileges in Unity Catalog"](https://docs.databricks.com/aws/en/data-governance/unity-catalog/manage-privileges/) doc covers every privilege type and object level in more depth than today's session needed.

## Self-Check
- [ ] I granted a privilege via the Catalog Explorer UI and confirmed it in `SHOW GRANTS`.
- [ ] I revoked that same privilege via SQL, and confirmed it disappeared.
- [ ] I granted it back via SQL, and confirmed it looked identical to the UI-granted version.
- [ ] I read real grants off the real `gbmart` catalog, schema, and table — read-only, no writes.

## What's Next
Head to `Day12_3_HOL2_Genie_Problem_Statement_Gold_Layer.html` — you'll build a real Genie Agent over `gbmart.gold`, and one of the questions you're pushed to ask it deliberately touches a governed/masked column, so you can watch everything from ILT 4 and today's HOL apply automatically, with zero extra setup.